# DSTS — LLM Inference Optimisation: 8B Model Benchmark (RunPod)

This notebook runs the **vocabulary-pruning** experiments on `meta-llama/Llama-3.1-8B-Instruct`
on a RunPod GPU pod. Speculative-decoding scripts are omitted (negative result on 1B).

## Prerequisites (do these ONCE before launching the pod)

### 1. HuggingFace token (for Llama-3.1-8B-Instruct access)
1. Go to https://huggingface.co/settings/tokens → create a token with **Read** scope
2. Accept the **Llama-3.1** license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
   (separate gate from Llama-3.2 — must be accepted even if you already accepted 3.2)
3. In RunPod: **Secrets** → **Add Secret**
   - Key: `HF_TOKEN`
   - Value: your HuggingFace token (starts with `hf_...`)

### 2. GitHub token (to clone the private repo)
1. GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)
   → Generate new token → scope: **repo** (full)
2. In RunPod: **Secrets** → **Add Secret**
   - Key: `GITHUB_TOKEN`
   - Value: your PAT (starts with `ghp_...`)
3. Reference both secrets in your pod template as `{{ RUNPOD_SECRET_HF_TOKEN }}` and
   `{{ RUNPOD_SECRET_GITHUB_TOKEN }}` — RunPod injects them as environment variables
   `RUNPOD_SECRET_HF_TOKEN` and `RUNPOD_SECRET_GITHUB_TOKEN` at pod startup.

### 3. GPU — two A100 / two L40S / or any pair with ≥32 GB combined VRAM
`Llama-3.1-8B-Instruct` in bfloat16 requires ~16 GB VRAM; two GPUs give headroom
for KV-cache and router overhead.

### 4. No pretrained artefacts needed
The `router_checkpoint.pt` and `mlp_transition_graph.pt` from the 1B run are **incompatible**
with the 8B model (hidden dim: 2048 → 4096). Both will be rebuilt from scratch.
This adds ~60 min to the dual-encoder + graph experiments.

---
**After running:** download `results.zip` from `/workspace/results.zip` via the RunPod
file browser or `scp`/`rsync`.  
**Locally:** unzip into your `dsts/results_8b/` folder, then run `python run_plots.py`

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — check pod GPU settings')

import torch
print(f'torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'CUDA device {i}: {torch.cuda.get_device_name(i)}, '
              f'{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: HuggingFace token + model selection ───────────────────────────────
# RunPod injects secrets as env vars with the RUNPOD_SECRET_ prefix.
# Reference them in your pod template as {{ RUNPOD_SECRET_HF_TOKEN }}.
import os

hf_token = os.environ.get('RUNPOD_SECRET_HF_TOKEN') or os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError(
        'HF_TOKEN not found.\n'
        'Set it in RunPod Secrets as HF_TOKEN and reference it in the pod template '
        'as {{ RUNPOD_SECRET_HF_TOKEN }}.'
    )

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token  # legacy env var
print('HF_TOKEN loaded ✓')

# Select the 8B model — all run_*.py scripts read MODEL_NAME from the environment
os.environ['MODEL_NAME'] = 'meta-llama/Llama-3.1-8B-Instruct'
print(f'MODEL_NAME set to: {os.environ["MODEL_NAME"]}')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
# RunPod PyTorch images ship with torch/CUDA; install the remaining packages.
!pip install -q \
    'transformers>=4.40.0' \
    'accelerate>=0.29.0' \
    'datasets>=2.18.0' \
    'sentencepiece>=0.2.0' \
    'tokenizers>=0.19.0' \
    'faiss-cpu' \
    'scikit-learn>=1.4.0' \
    'sentence-transformers>=2.7.0' \
    'tqdm>=4.66.0' \
    'numpy>=1.26.0' \
    'pandas>=2.2.0' \
    'matplotlib>=3.8.0' \
    'seaborn>=0.13.0'

print('Dependencies installed ✓')

In [ ]:
# ── Cell 4: Clone repo (private) ─────────────────────────────────────────────
# Reads GITHUB_TOKEN from the RunPod secret injected as RUNPOD_SECRET_GITHUB_TOKEN.
import os

gh_token = os.environ.get('RUNPOD_SECRET_GITHUB_TOKEN') or os.environ.get('GITHUB_TOKEN')
if not gh_token:
    raise RuntimeError(
        'GITHUB_TOKEN not found.\n'
        'Set it in RunPod Secrets as GITHUB_TOKEN and reference it in the pod template '
        'as {{ RUNPOD_SECRET_GITHUB_TOKEN }}.'
    )

WORKDIR = '/workspace/dsts'
REPO_URL = f'https://{gh_token}@github.com/deil87/dsts.git'

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {WORKDIR} remote set-url origin {REPO_URL}
    !git -C {WORKDIR} pull

%cd {WORKDIR}
print(f'Working directory: {os.getcwd()}')

# Re-export after %cd (subshells inherit env from the kernel)
os.environ['MODEL_NAME'] = 'meta-llama/Llama-3.1-8B-Instruct'
print(f'MODEL_NAME: {os.environ["MODEL_NAME"]}')
!ls -la

In [ ]:
# ── Cell 5: Artefacts note ────────────────────────────────────────────────────
# The 1B router_checkpoint.pt (hidden_dim=2048) and mlp_transition_graph.pt are
# INCOMPATIBLE with Llama-3.1-8B-Instruct (hidden_dim=4096). They will be
# rebuilt from scratch during run_dual_encoder.py (~50 min) and run_graph.py (~15 min).
#
# If you have 8B-compatible artefacts pre-built, copy them into /workspace/dsts/results/
# before running the experiment cells below.
import os
os.makedirs('results', exist_ok=True)

for fname in ['router_checkpoint.pt', 'mlp_transition_graph.pt']:
    path = f'results/{fname}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'Found artefact: {path} ({size_mb:.1f} MB)')
        print('  WARNING: verify this is an 8B checkpoint (hidden_dim=4096); 1B checkpoints are incompatible.')
    else:
        print(f'No artefact found at {path} — will build from scratch (expected).')

In [ ]:
# ── Cell 6: Run baseline ────────────────────────────────────────────────────────
# Expected runtime: ~15 min (8B on 2-GPU pod)
print('=' * 60)
print('EXPERIMENT 1/6: Baseline')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_baseline.py

In [ ]:
# ── Cell 7: Static + Cosine router ──────────────────────────────────────────────
# Expected runtime: ~40 min (8B on 2-GPU pod)
print('=' * 60)
print('EXPERIMENT 2/6: Static + Cosine Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_static.py

In [ ]:
# ── Cell 8: Cluster router ──────────────────────────────────────────────────────
# Expected runtime: ~25 min (8B on 2-GPU pod)
print('=' * 60)
print('EXPERIMENT 3/6: Cluster Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_cluster.py

In [ ]:
# ── Cell 9: Dual-encoder router ─────────────────────────────────────────────────
# Expected runtime: ~90 min (8B on 2-GPU pod, includes ~50 min training)
# Fast path: Triton-fused MLP + GPU-native token union (no CPU round-trips).
print('=' * 60)
print('EXPERIMENT 4/6: Dual-Encoder Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_dual_encoder.py

In [ ]:
# ── Cell 10: MLP transition graph router ────────────────────────────────────────
# Expected runtime: ~45 min (8B on 2-GPU pod, includes ~15 min graph build)
print('=' * 60)
print('EXPERIMENT 5/6: MLP Transition Graph Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_graph.py

In [ ]:
# ── Cell 11: Plots ──────────────────────────────────────────────────────────────
# Expected runtime: ~1 min
print('=' * 60)
print('EXPERIMENT 6/6: Generate Plots')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_plots.py

In [ ]:
# ── Cell 12: Summary of results ───────────────────────────────────────────────
import pandas as pd
import os

print('Results directory contents:')
for f in sorted(os.listdir('results')):
    size = os.path.getsize(f'results/{f}')
    print(f'  {f:<50s} {size/1024:>8.1f} KB')

# Print the consolidated results table if it exists
all_csv = 'results/all_results.csv'
if os.path.exists(all_csv):
    print('\n=== All Results ===')
    df = pd.read_csv(all_csv)
    with pd.option_context('display.max_rows', 50, 'display.max_columns', 20,
                           'display.width', 120, 'display.float_format', '{:.3f}'.format):
        print(df.to_string(index=False))

In [ ]:
# ── Cell 13: Package results for download ─────────────────────────────────────
# Zips the entire results/ directory → /workspace/results.zip
# Download via the RunPod file browser, or:
#   scp root@<pod-ip>:/workspace/results.zip .
#   rsync -avz root@<pod-ip>:/workspace/results.zip .
import shutil, os

OUTPUT_ZIP = '/workspace/results'
shutil.make_archive(OUTPUT_ZIP, 'zip', 'results')

zip_size = os.path.getsize(OUTPUT_ZIP + '.zip') / 1e6
print(f'results.zip created: {zip_size:.1f} MB  →  /workspace/results.zip')
print()
print('To download from your local machine:')
print('  scp root@<pod-ip>:/workspace/results.zip .')
print()
print('Locally, unzip into your dsts/results_8b/ folder:')
print('  mkdir -p results_8b')
print('  unzip results.zip -d results_8b/')
print('  RESULTS_DIR=results_8b python run_plots.py')